In [ ]:
from pathlib import Path
import json, os, shlex, subprocess, sys

def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'run_fast_causal_baselines.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the package root')

PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
RUNNER_ENV['PYTHONUNBUFFERED'] = '1'
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (str(PACKAGE_ROOT / 'src'), RUNNER_ENV.get('PYTHONPATH'))))
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))
from calcium_transient_rising_flank.checkpointing import format_progress

# Environment setup: uv sync --frozen --all-extras
RUN_VAR_GRANGER = True
OUTPUT_ROOT = PACKAGE_ROOT / 'outputs/matched_motorneuron_benchmark'
OUTPUT_DIR = OUTPUT_ROOT / 'var-granger'
command = [
    RUNNER_PYTHON, 'examples/run_fast_causal_baselines.py',
    '--dataset', 'motorneurons', '--algorithm', 'var-granger',
    '--output-dir', str(OUTPUT_DIR), '--case-data-dir', 'data/motoneurons',
    '--cases', 'C,D', '--recordings', 'F3T1,F3T2,F5T2',
    '--representations', 'rise,fall', '--max-lag', '1', '--resume',
]
print(f'[notebook] launching: {shlex.join(command)}', flush=True)
if RUN_VAR_GRANGER:
    subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
state_path = OUTPUT_DIR / 'motorneurons_var-granger_resume.json'
if state_path.exists():
    state = json.loads(state_path.read_text())
    print(format_progress(int(state.get('completed_units', 0)), max(1, int(state.get('total_units', 1))), label='VAR-Granger motorneuron fits'))
summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    print(summary_path.read_text())
required = [OUTPUT_ROOT / algorithm / filename for algorithm in ('cgc', 'cgc-star', 'pcmciplus', 'var-granger') for filename in ('baseline_rows.csv', 'summary.json')]
analysis_command = [RUNNER_PYTHON, 'examples/analyze_matched_motorneuron_benchmark.py', '--input-root', str(OUTPUT_ROOT), '--output-dir', str(OUTPUT_ROOT / 'analysis')]
print(f'[notebook] analysis command: {shlex.join(analysis_command)}', flush=True)
if RUN_VAR_GRANGER and all(path.is_file() for path in required):
    subprocess.run(analysis_command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
else:
    print('[notebook] analysis waits until all four matched algorithms are complete')


## Empirical sensitivity corresponding to H1–H4

The synthetic H1–H4 recovery tests are executed in `notebooks/simulations/var_granger.ipynb`, where edge truth is known. This empirical notebook does not relabel graph density as accuracy. It reports matched case/recording inputs, edge density, bilateral summaries, and c-GC/c-GC* conditioning-depth sensitivity only.


In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
analysis_dir = OUTPUT_ROOT / 'analysis'
if (analysis_dir / 'analysis-report.md').is_file():
    display(Markdown((analysis_dir / 'analysis-report.md').read_text()))
    display(Markdown((analysis_dir / 'stats-appendix.md').read_text()))
    display(pd.read_csv(analysis_dir / 'descriptive_paired_contrasts.csv').round(4))
    display(Image(filename=str(analysis_dir / 'matched_motorneuron_graph_density.png')))
    display(Image(filename=str(analysis_dir / 'motorneuron_cgc_conditioning_depth_sensitivity.png')))
else:
    print('[notebook] empirical analysis will appear after all four algorithms finish')

## Reviewer audit: empirical signed-input and cross-representation sensitivity

This run compares signed and rectified inputs and evaluates source fall residuals against target rises. These are descriptive sensitivity checks because the recordings do not provide signed or cross-phase graph truth.


In [ ]:
RUN_REVIEWER_AUDITS = True
REVIEW_ROOT = PACKAGE_ROOT / 'outputs/representation_bias_benchmark/motorneurons'
review_command = [RUNNER_PYTHON, 'examples/run_representation_bias_benchmarks.py', '--dataset', 'motorneurons', '--algorithm', 'var-granger', '--output-dir', str(REVIEW_ROOT / 'var-granger'), '--case-data-dir', 'data/motoneurons', '--cases', 'C,D', '--recordings', 'F3T1,F3T2,F5T2', '--n-surrogates', '6000', '--resume']
print(f'[notebook] reviewer-audit command: {shlex.join(review_command)}', flush=True)
if RUN_REVIEWER_AUDITS:
    subprocess.run(review_command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
required = [REVIEW_ROOT / method / filename for method in ('cgc', 'cgc-star', 'pcmciplus', 'var-granger') for filename in ('signed_information_rows.csv', 'cross_representation_rows.csv', 'summary.json')]
review_analysis = [RUNNER_PYTHON, 'examples/analyze_representation_bias_benchmarks.py', '--dataset', 'motorneurons', '--input-root', str(REVIEW_ROOT), '--output-dir', str(REVIEW_ROOT / 'analysis'), '--cases', 'C,D', '--recordings', 'F3T1,F3T2,F5T2']
if RUN_REVIEWER_AUDITS and all(path.is_file() for path in required):
    subprocess.run(review_analysis, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
else:
    print('[notebook] common reviewer analysis waits for all four methods')
review_summary_path = REVIEW_ROOT / 'analysis/summary.json'
if review_summary_path.is_file():
    print(review_summary_path.read_text())
